In [ ]:
%load_ext blackcellmagic 
# %black -l 120
%load_ext autoreload
%autoreload 2

In [ ]:

config_42px_c24c24 = {
    "n_actions": 4,
    "batch_size": 16,
    "architectures": ["cnn"], #[,"impala"]
    "feature_list": [[24, 24]],
    "gap_list": [True],
    "layer_norm": ([1,1], True),
    "low_scale": False,
    "n_conv": 2,
    "n_fc": 1,
}
config_24px_c16c24c24 = {
    "n_actions": 4,
    "batch_size": 16,
    "architectures": ["cnn"], #[,"impala"]
    "feature_list": [[16, 24, 24]],
    "gap_list": [True],
    "layer_norm": ([0,1,0], True),
    "low_scale": True,
    "n_conv": 3,
    "n_fc": 1,
}

config_MINATAR_PXL42_K4S2_c24_f128 = {
    "n_actions": 4,
    "batch_size": 16,
    "architectures": ["cnn"], #[,"impala"]
    "feature_list": [[24, 128]],
    "gap_list": [False],
    "layer_norm": ([1], True),
    "low_scale": False,
    "n_conv": 1,
    "n_fc": 2,
}

config_MINATAR_PXL24_K5S1_c24_f128 = {
    "n_actions": 4,
    "batch_size": 16,
    "architectures": ["cnn"], #[,"impala"]
    "feature_list": [[24, 128]],
    "gap_list": [False],
    "layer_norm": ([1], True),
    "low_scale": True,
    "n_conv": 1,
    "n_fc": 2,
}
config_MINATAR_PXL42_K4S2_c16_f128 = {
    "n_actions": 4,
    "batch_size": 16,
    "architectures": ["cnn"], #[,"impala"]
    "feature_list": [[16, 128]],
    "gap_list": [False],
    "layer_norm": ([1], True),
    "low_scale": False,
    "n_conv": 1,
    "n_fc": 2,
}

config_MINATAR_PXL42_K4S2_c16_f64 = {
    "n_actions": 4,
    "batch_size": 16,
    "architectures": ["cnn"], #[,"impala"]
    "feature_list": [[24, 64]],
    "gap_list": [False],
    "layer_norm": ([1], True),
    "low_scale": False,
    "n_conv": 1,
    "n_fc": 2,
}
# config_1 = {
#     "n_actions": 4,
#     "batch_size": 16,
#     "architectures": ["cnn"], #[,"impala"]
#     "feature_list": [[24, 24]],
#     "gap_list": [True],
#     "layer_norm": (True, True),
#     "low_scale": False,
#     "n_conv": 2,
#     "n_fc": 1,
# }
#
# config_2 = {
#     "n_actions": 4,
#     "batch_size": 16,
#     "architectures": ["cnn"], #[,"impala"]
#     "feature_list": [[16, 24, 24]],
#     "gap_list": [True],
#     "layer_norm": (True, True),
#     "low_scale": True,
#     "n_conv": 3,
#     "n_fc": 1,
# }


In [ ]:
import os
#reproducability and determinism
os.environ["XLA_FLAGS"] = (
    #"--xla_gpu_autotune_level=0 "
    #"--xla_gpu_deterministic_ops=true "
    "--xla_backend_optimization_level=0 "
)

from slimdqn.algorithms.dqn import DQN
from slimdqn.algorithms.dqnrcshared import DQNRCShared
from slimdqn.algorithms.idqnshared import iDQNShared
from slimdqn.algorithms.gidqnshared import GiDQNShared


import jax
import jax.numpy as jnp
from tests.utils import Generator

def run(n_actions, batch_size, architectures, feature_list, gap_list, layer_norm, low_scale, n_conv, n_fc):
    def count_params(params):
        return sum(x.size for x in jax.tree.leaves(params))


    def count_flops(q, has_target_params=False):
        best_action_compiled = (
            jax.jit(q.best_action).lower(q.params, sample_generator.state(jax.random.PRNGKey(0))).compile()
        )
        if not has_target_params:
            learn_on_batch_compiled = (
                jax.jit(q.learn_on_batch)
                .lower(q.params, q.optimizer_state, sample_generator.samples(jax.random.PRNGKey(0)), jnp.ones(batch_size))
                .compile()
            )
        else:
            learn_on_batch_compiled = (
                jax.jit(q.learn_on_batch)
                .lower(
                    q.params,
                    q.target_params,
                    q.optimizer_state,
                    sample_generator.samples(jax.random.PRNGKey(0)),
                    jnp.ones(batch_size),
                )
                .compile()
            )

        return best_action_compiled, learn_on_batch_compiled

    pixels_frame_stack= (24, 24, 2) if low_scale else (42, 42, 2) #pixel x pixel, frame stack
    sample_generator = Generator(batch_size, pixels_frame_stack, n_actions)


    metrics = {}
    metrics["flops"] = {}
    metrics["num_params"] = {}

    for idx, architecture in enumerate(architectures):
        features = feature_list[idx]
        gap = gap_list[idx]
        print(f"--- DQN - {architecture} ---")
        q_dqn = DQN(
            jax.random.PRNGKey(0),
            observation_dim= pixels_frame_stack,
            n_actions= n_actions,
            features= features,
            architecture_type= architecture,
            layer_norm = layer_norm,
            gap=gap,
            learning_rate= 6.25e-5,
            gamma= 0.99,
            update_horizon= 1,
            update_to_data= 0.25,
            target_update_period= 8000,
            low_scale=low_scale,
            n_conv= n_conv ,
            n_fc=n_fc,
        )
        metrics["num_params"][f"dqn_{architecture}"] = count_params(q_dqn.params) + count_params(q_dqn.target_params)
        q_dqn_best_action_compiled, q_dqn_learn_on_batch_compiled = count_flops(q_dqn, has_target_params=True)
        metrics["flops"][f"dqn_{architecture}"] = q_dqn_learn_on_batch_compiled.cost_analysis()[0]["flops"]
        print("DQN Num params: ", metrics["num_params"][f"dqn_{architecture}"])
        print("DQN FLOPs best action: ", q_dqn_best_action_compiled.cost_analysis()[0]["flops"])
        print("DQN FLOPs to learn on a batch: ", metrics["flops"][f"dqn_{architecture}"], "\n")

        total_flops_dqn =q_dqn_best_action_compiled.cost_analysis()[0]["flops"] + metrics["flops"][f"dqn_{architecture}"]
        print("Linear Flops both:", total_flops_dqn, "\n")



        # print(f"--- DQNRC - {architecture} ---")
        # q_qrc = DQNRCShared(
        #     jax.random.PRNGKey(0),
        #     observation_dim= pixels_frame_stack,
        #     n_actions= n_actions,
        #     features= features,
        #     architecture_type= architecture,
        #     layer_norm = layer_norm,
        #     gap=gap,
        #     linear_heads=True,
        #     learning_rate= 6.25e-5,
        #     gamma= 0.99,
        #     update_horizon= 1,
        #     update_to_data= 0.25,
        #     target_update_period= 8000,
        #     weight_decay= 1,
        #     low_scale=low_scale,
        #     n_conv= n_conv ,
        #     n_fc=n_fc,
        # )
        # metrics["num_params"][f"qrc_{architecture}"] = count_params(q_qrc.params)
        # q_qrc_best_action_compiled, q_qrc_learn_on_batch_compiled = count_flops(q_qrc, has_target_params=False)
        # metrics["flops"][f"qrc_{architecture}"] = q_qrc_learn_on_batch_compiled.cost_analysis()[0]["flops"]
        # print("DQNRC with linear heads params: ", metrics["num_params"][f"qrc_{architecture}"])
        # print("DQNRC FLOPs best action: ", q_qrc_best_action_compiled.cost_analysis()[0]["flops"])
        # print("DQNRC FLOPs to learn on a batch: ", metrics["flops"][f"qrc_{architecture}"], "\n")
        #
        # print("Linear Flops both:", q_qrc_best_action_compiled.cost_analysis()[0]["flops"] + metrics["flops"][f"qrc_{architecture}"], "\n")
        #
        # print(f"--- i-DQN - {architecture} ---")
        # q_idqn = iDQNShared(
        #     jax.random.PRNGKey(0),
        #     observation_dim= pixels_frame_stack,
        #     n_actions= n_actions,
        #     n_bellman_iterations=5,
        #     features= features,
        #     architecture_type= architecture,
        #     layer_norm = layer_norm,
        #     gap=gap,
        #     linear_heads=True,
        #     learning_rate= 6.25e-5,
        #     gamma= 0.99,
        #     update_horizon= 1,
        #     update_to_data= 0.25,
        #     target_update_period= 8000,
        #     low_scale=low_scale,
        #     n_conv= n_conv ,
        #     n_fc=n_fc,
        # )
        # metrics["num_params"][f"idqn_{architecture}"] = count_params(q_idqn.params) + count_params(q_idqn.target_params)
        # q_idqn_best_action_compiled, q_idqn_learn_on_batch_compiled = count_flops(q_idqn, has_target_params=True)
        # metrics["flops"][f"idqn_{architecture}"] = q_idqn_learn_on_batch_compiled.cost_analysis()[0]["flops"]
        # print("iDQN with linear heads params: ", metrics["num_params"][f"idqn_{architecture}"])
        # print("Linear i-DQN FLOPs best action: ", q_idqn_best_action_compiled.cost_analysis()[0]["flops"])
        # print("Linear i-DQN FLOPs to learn on a batch: ", metrics["flops"][f"idqn_{architecture}"], "\n")
        #
        # print("Linear Flops both:", q_idqn_best_action_compiled.cost_analysis()[0]["flops"] + metrics["flops"][f"idqn_{architecture}"], "\n")
        #
        # print(f"--- Gi-DQN - {architecture} ---")
        # q_gidqn = GiDQNShared(
        #     jax.random.PRNGKey(0),
        #     observation_dim= pixels_frame_stack,
        #     n_actions= n_actions,
        #     n_bellman_iterations=5,
        #     features= features,
        #     architecture_type= architecture,
        #     layer_norm = layer_norm,
        #     gap=gap,
        #     linear_heads=True,
        #     learning_rate= 6.25e-5,
        #     gamma= 0.99,
        #     update_horizon= 1,
        #     update_to_data= 0.25,
        #     target_update_period= 8000,
        #     weight_decay= 1,
        #     low_scale=low_scale,
        #     n_conv= n_conv ,
        #     n_fc=n_fc,
        # )
        # metrics["num_params"][f"gidqn_{architecture}"] = count_params(q_gidqn.params) + count_params(q_gidqn.target_params)
        # q_gidqn_best_action_compiled, q_gidqn_learn_on_batch_compiled = count_flops(q_gidqn, has_target_params=True)
        # metrics["flops"][f"gidqn_{architecture}"] = q_gidqn_learn_on_batch_compiled.cost_analysis()[0]["flops"]
        # print("GiDQN Num params with linear heads: ", metrics["num_params"][f"gidqn_{architecture}"])
        # print("Linear FLOPs best action: ", q_gidqn_best_action_compiled.cost_analysis()[0]["flops"])
        # print("Linear FLOPs to learn on a batch: ", metrics["flops"][f"gidqn_{architecture}"], "\n")
        # print("Linear Flops both:", q_gidqn_best_action_compiled.cost_analysis()[0]["flops"] + metrics["flops"][f"gidqn_{architecture}"], "\n")

        return total_flops_dqn


    print(metrics)

In [ ]:
print("-------- config_42px_c24c24 -----------------------")
total_flops_dqn_1 = run(**config_42px_c24c24)

print("-------- config_24px_c16c24c24 -----------------------")
total_flops_dqn_2 = run(**config_24px_c16c24c24)



print("---------- MINATAR  42 ----------------------")
total_flops_dqn_3 = run(**config_MINATAR_PXL42_K4S2_c24_f128)

print("---------- MINATAR 24  ----------------------")
total_flops_dqn_4 = run(**config_MINATAR_PXL24_K4S2_c24_f128)

print("---------- MINATAR f128  ----------------------")
total_flops_dqn_5 = run(**config_MINATAR_PXL42_K4S2_c16_f128)

print("---------- MINATAR f64  ----------------------")
total_flops_dqn_6 = run(**config_MINATAR_PXL42_K4S2_c16_f64)

In [ ]:
import matplotlib.pyplot as plt

# Data extracted from your logs
labels = ['C24-C24;42Px', 'C16-C24-C24;24Px', 'MA-42PxK4S2','MA-24PxK5S1', 'MA-42PxK4S2C16', 'TRY']
values = [total_flops_dqn_1, total_flops_dqn_2, total_flops_dqn_3, total_flops_dqn_4, total_flops_dqn_5,total_flops_dqn_6]

fig, ax = plt.subplots(figsize=(8, 6))

bars = ax.bar(labels, values, color=['#1f77b4', '#ff7f0e', 'red', 'purple', 'cyan', 'cyan'])

ax.set_title('Total FLOPs Comparison (Best Action + Learn on Batch)')
ax.set_ylabel('Total FLOPs')

ax.ticklabel_format(style='plain', axis='y')

for bar in bars:
    yval = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        yval + (max(values) * 0.02),
        f'{yval:,.0f}',
        ha='center',
        va='bottom',
        fontweight='bold'
    )

plt.tight_layout()

plt.savefig('flops_comparison_MINATAR_c24_f128_PXL24_vs_42___MINATARc16_c128_PXL42_K4S2.png', dpi=300)
plt.show()